In [ ]:
import glob
import os
import sqlite3
import pandas as pd
import numpy as np
import networkx as nx
from collections import Counter
import matplotlib.pyplot as plt
from tqdm.notebook import trange, tqdm
import itertools
from collections import defaultdict

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns

plt.style.use('ggplot')

rcParams['font.family'] = 'sans-serif'
rcParams['font.style'] = 'normal'

rcParams['figure.facecolor'] = 'white'
rcParams['figure.dpi'] = 150

rcParams['savefig.bbox'] = 'tight'
rcParams['savefig.dpi'] = 300
rcParams['savefig.transparent'] = True

rcParams['axes.spines.right'] = False
rcParams['axes.spines.top'] = False
rcParams['axes.labelsize'] = 10
rcParams['axes.labelcolor'] = 'black'
rcParams['axes.edgecolor'] = 'black'
rcParams['axes.linewidth'] = 1
rcParams['axes.facecolor'] = 'white'

rcParams['legend.fontsize'] = 6

rcParams['xtick.color'] = 'black'
rcParams['ytick.color'] = 'black'
rcParams['xtick.major.width'] = 1
rcParams['ytick.major.width'] = 1
rcParams['xtick.major.size'] = 2
rcParams['ytick.major.size'] = 2
rcParams['xtick.labelsize'] = 8
rcParams['ytick.labelsize'] = 8

rcParams['lines.linewidth'] = 1
rcParams['lines.markersize'] = 5

rcParams['grid.color'] = 'white'
rcParams['grid.linewidth'] = 0.0

# Define settings and runs

In [ ]:
networks = ['BA', 'ER']
recsyss = ['F', 'FP', 'P', 'RC']
runs = list(range(10))  # 0-9

# Average number of recommendations

In [ ]:
# data_for_plots[(network,recsys)][round] = [counts across runs]
data_for_plots = defaultdict(lambda: defaultdict(list))

pbar = tqdm(
    total=len(networks)*len(recsyss)*len(runs),
    desc="Processing Experiments",
    unit="experiment"
)

for network in networks:
    for recsys in recsyss:

        key = (network, recsys)

        for run in runs:

            db_path = f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"

            with sqlite3.connect(db_path) as conn:
                cursor = conn.cursor()

                cursor.execute("""
                    SELECT round, COUNT(*)
                    FROM recommendations
                    GROUP BY round
                """)

                # fetch aggregated results only
                for r, count in cursor.fetchall():
                    data_for_plots[key][r].append(count)

            pbar.update(1)

        # compute stats AFTER all runs for this (network,recsys)
        rounds = list(data_for_plots[key].keys())

        avg = {r: np.mean(data_for_plots[key][r]) for r in rounds}
        std = {r: np.std(data_for_plots[key][r]) for r in rounds}

        data_for_plots[key]['avg'] = avg
        data_for_plots[key]['std'] = std

pbar.close()


In [ ]:
fig, axes = plt.subplots(2, sharey=True, figsize=(12, 8))

for i, network in enumerate(networks):
    for recsys in recsyss:
        ax = axes[i]
        rounds = np.array(list(data_for_plots[(network, recsys)]['avg'].keys()))
        avg_counts = np.array(list(data_for_plots[(network, recsys)]['avg'].values()))
        std_counts = np.array(list(data_for_plots[(network, recsys)]['std'].values()))
        
        cum_counts = np.cumsum(avg_counts)
        
        ax.plot(rounds, cum_counts, label=f'{recsys} (Avg)')
        ax.fill_between(rounds, cum_counts - std_counts, cum_counts + std_counts, alpha=0.2)
        ax.set_xlabel('Round')
        ax.set_ylabel('Average Number of Recommendations')
        ax.legend()
plt.show()

del data_for_plots  # free memory

# How many times individual posts are recommended?

In [ ]:
import gc

pbar = tqdm(
    total=len(networks)*len(recsyss)*len(runs),
    desc="Processing Experiments",
    unit="experiment"
)

avg_data = {}

for network in networks:
    for recsys in recsyss:

        key = (network, recsys)
        print(f"Processing {network} - {recsys}")

        sum_counts = Counter()      # sum of counts per k across runs
        sum_squares = Counter()     # sum of squares for std calculation
        all_k = set()               # keep track of all k encountered

        for run in runs:

            db_path = f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"

            # --- Step 1: Count recommendations per post in this run ---
            post_counter = Counter()  # post_id -> # of recommendations

            with sqlite3.connect(db_path) as conn:
                cursor = conn.cursor()
                cursor.execute("SELECT post_ids FROM recommendations")
                
                while True:
                    rows = cursor.fetchmany(10000)  # fetch in batches
                    if not rows:
                        break
                    for (post_ids,) in rows:
                        for pid in post_ids.split("|"):
                            post_counter[int(pid)] += 1

            # --- Step 2: Convert per-post counts into a histogram ---
            hist = Counter(post_counter.values())

            # update sums for mean/std
            for k, v in hist.items():
                sum_counts[k] += v
                sum_squares[k] += v**2
                all_k.add(k)

            pbar.update(1)

            # free memory for this run
            del post_counter, hist
            gc.collect()

        # --- Step 3: Compute average and std histogram across runs ---
        avg_hist = {}
        std_hist = {}
        num_runs = len(runs)

        for k in sorted(all_k):
            avg_hist[k] = sum_counts[k] / num_runs
            std_hist[k] = np.sqrt(sum_squares[k] / num_runs - avg_hist[k]**2)

        # --- Step 4: Store results for plotting ---
        print('storing results for plotting')
        avg_data[key] = {
            "avg_hist": avg_hist,
            "std_hist": std_hist
            # run_hists removed to save memory
        }

print("Done computing histograms and averages.")


In [ ]:
fig, axes = plt.subplots(2, sharey=True, figsize=(12, 8))

for i, network in enumerate(networks):
    ax = axes[i]

    for recsys in recsyss:
        key = (network, recsys)
        hist = avg_data[key]["avg_hist"]
        std_hist = avg_data[key]["std_hist"]

        ks = np.array(sorted(hist.keys()))
        vals = np.array([hist[k] for k in ks])
        errs = np.array([std_hist[k] for k in ks])

        # main line
        line = ax.plot(ks, vals, marker='o', label=recsys)[0]

        # shaded std area
        ax.fill_between(
            ks,
            vals - errs,
            vals + errs,
            color=line.get_color(),
            alpha=0.2
        )

    ax.set_title(f"Network: {network}")
    ax.set_xlabel("Recommendations per post (k)")
    ax.set_ylabel("# Posts")
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, sharey=True, figsize=(12, 8))

for i, network in enumerate(networks):
    ax = axes[i]

    for recsys in recsyss:
        k = (network, recsys)

        ks = sorted(avg_data[k]["avg_hist"].keys())
        vals = [avg_data[k]["avg_hist"][x] for x in ks]

        ax.loglog(ks, vals, 'o-', label=recsys, alpha=0.7)

    ax.set_title(network)
    ax.set_xlabel("Recommendations per post (k)")
    ax.set_ylabel("# Posts")
    ax.legend()

plt.tight_layout()
plt.show()


# How many posts are recommended in two consecutive rounds?

In [ ]:
import sqlite3
import pandas as pd
from collections import defaultdict

# Store per-run consecutive recommendation probabilities
consec_probs_runs = defaultdict(list)  # (network, recsys) -> list of per-run probabilities

for network in networks:
    for recsys in recsyss:
        key = (network, recsys)

        for run in runs:
            db_path = f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"

            # --- Collect rounds per post ---
            post_rounds = defaultdict(list)  # post_id -> list of rounds

            with sqlite3.connect(db_path) as conn:
                cursor = conn.cursor()
                cursor.execute("SELECT round, post_ids FROM recommendations")
                for r, post_ids in cursor:
                    for pid in post_ids.split("|"):
                        post_rounds[int(pid)].append(r)

            # --- Compute probability of consecutive recommendation for this run ---
            consec_count = 0
            total_pairs = 0

            for rounds in post_rounds.values():
                rounds_sorted = sorted(rounds)
                total_pairs += max(len(rounds_sorted) - 1, 0)
                for i in range(1, len(rounds_sorted)):
                    if rounds_sorted[i] == rounds_sorted[i-1] + 1:
                        consec_count += 1

            prob = consec_count / total_pairs if total_pairs > 0 else 0
            consec_probs_runs[key].append(prob)

# --- Build DataFrame with average ± std ---
rows = []
for key, probs in consec_probs_runs.items():
    network, recsys = key
    avg_prob = sum(probs) / len(probs)
    std_prob = pd.Series(probs).std()
    rows.append({
        "network": network,
        "recsys": recsys,
        "avg_prob_consec": avg_prob,
        "std_prob_consec": std_prob
    })

df_consec = pd.DataFrame(rows)
df_consec

# How many rounds pass between two recommendations?

In [ ]:
consec_data = {}


pbar = tqdm(
    total=len(networks)*len(recsyss)*len(runs),
    desc="Processing Experiments",
    unit="experiment"
)

for network in networks:
    for recsys in recsyss:

        key = (network, recsys)

        # To accumulate counts across runs
        consec_counts_acc = defaultdict(int)      # number of consecutive rounds a post is recommended
        inter_event_counts_acc = defaultdict(int) # gaps between recommendations

        total_posts_consec = 0
        total_events_inter = 0

        for run in runs:
            db_path = f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"

            # --- Step 1: load recommendations per post per round ---
            post_rounds = defaultdict(list)  # post_id -> list of rounds it was recommended

            with sqlite3.connect(db_path) as conn:
                cursor = conn.cursor()
                cursor.execute("SELECT round, post_ids FROM recommendations")

                for r, post_ids in cursor:
                    for pid in post_ids.split("|"):
                        post_rounds[int(pid)].append(r)

            # --- Step 2: for each post, compute consecutive recommendations & inter-event gaps ---
            for pid, rounds in post_rounds.items():
                rounds_sorted = sorted(rounds)

                # consecutive recommendations
                for i in range(1, len(rounds_sorted)):
                    if rounds_sorted[i] == rounds_sorted[i-1] + 1:
                        consec_counts_acc[1] += 1
                    else:
                        # can track longer consecutive streaks if needed
                        pass

                total_posts_consec += len(rounds_sorted) - 1  # total pairs to normalize

                # inter-event gaps
                for i in range(1, len(rounds_sorted)):
                    gap = rounds_sorted[i] - rounds_sorted[i-1] - 1
                    inter_event_counts_acc[gap] += 1
                    total_events_inter += 1

            pbar.update(1)
        
        # --- Step 3: convert counts to probabilities ---
        p_consec = {k: v / total_posts_consec for k, v in consec_counts_acc.items()}
        p_inter_event = {x: inter_event_counts_acc[x] / total_events_inter for x in inter_event_counts_acc}

        consec_data[key] = {
            "p_consecutive": p_consec,
            "p_inter_event": p_inter_event
        }

print("Done computing consecutive and inter-event probabilities.")


In [ ]:
fig, axes = plt.subplots(2, sharey=True, figsize=(12,6))
for i, network in enumerate(networks):
    ax = axes[i]
    for recsys in recsyss:
        key = (network, recsys)
        data = consec_data[key]["p_inter_event"]
        xs = sorted(data.keys())
        ys = [data[x] for x in xs]
        ax.plot(xs, ys, marker='o', label=recsys)
    ax.set_title(f"{network}: Probability of gap x between recommendations")
    ax.set_xlabel("Number of rounds without recommendation (x)")
    ax.set_ylabel("Probability")
    ax.set_yscale('log')  # optional, good for heavy tails
    ax.set_xscale('log')
    ax.legend()
plt.tight_layout()
plt.show()


# What percentage of posts is recommended in X consecutive rounds?

In [ ]:
# Store counts: (network, recsys) -> {streak_length: avg #posts across runs}
consec_streak_data = {}

pbar = tqdm(
    total=len(networks)*len(recsyss)*len(runs),
    desc="Processing Experiments",
    unit="experiment"
)


for network in networks:
    for recsys in recsyss:
        key = (network, recsys)

        # list of histograms per run
        run_streak_hists = []

        print(f"Processing {network} - {recsys}")

        for run in runs:
            db_path = f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"

            # collect rounds per post
            post_rounds = defaultdict(list)
            with sqlite3.connect(db_path) as conn:
                cursor = conn.cursor()
                cursor.execute("SELECT round, post_ids FROM recommendations")
                for r, post_ids in cursor:
                    for pid in post_ids.split("|"):
                        post_rounds[int(pid)].append(r)

            # compute consecutive streaks for each post
            streak_hist = Counter()  # streak_length -> #posts with this streak length

            for rounds in post_rounds.values():
                rounds_sorted = sorted(rounds)

                streak = 0
                prev_round = None

                for r in rounds_sorted:
                    if prev_round is None:
                        streak = 1
                    elif r == prev_round + 1:
                        streak += 1
                    else:
                        streak_hist[streak] += 1
                        streak = 1
                    prev_round = r

                # count last streak
                streak_hist[streak] += 1

            run_streak_hists.append(streak_hist)
            
            pbar.update(1)

        # --- average histogram across runs ---
        all_streaks = set().union(*run_streak_hists)
        avg_streak_hist = {}
        std_streak_hist = {}

        for s in all_streaks:
            counts = [h.get(s, 0) for h in run_streak_hists]
            avg_streak_hist[s] = np.mean(counts)
            std_streak_hist[s] = np.std(counts)

        consec_streak_data[key] = {
            "avg_hist": avg_streak_hist,
            "std_hist": std_streak_hist
        }
        

print("Done computing consecutive streak histograms.")


In [ ]:
fig, axes = plt.subplots(2, sharey=True, figsize=(12,6))

for i, network in enumerate(networks):
    ax = axes[i]
    n_recsys = len(recsyss)
    
    for j, recsys in enumerate(recsyss):
        key = (network, recsys)
        hist = consec_streak_data[key]["avg_hist"]
        std_hist = consec_streak_data[key]["std_hist"]
        ks = np.array(sorted(hist.keys()), dtype=float)
        vals = np.array([hist[k] for k in ks])
        errs = np.array([std_hist[k] for k in ks])

        # --- convert to percentage ---
        total_posts = vals.sum()
        vals_pct = vals / total_posts * 100
        errs_pct = errs / total_posts * 100

        # --- add horizontal jitter ---

        line = ax.plot(ks + (j*0.05), vals_pct, 'o-', label=recsys, alpha=0.5)[0]
        ax.fill_between(ks + (j*0.05), vals_pct-errs_pct, vals_pct+errs_pct, color=line.get_color(), alpha=0.5)

    ax.set_title(f"Network: {network}")
    ax.set_xlabel("Consecutive recommendation streak length")
    ax.set_ylabel("% Posts")
    ax.legend()

plt.tight_layout()
plt.show()


# What is the average number -- of the total posts -- that is recommended in each round?

In [ ]:
# Store per-run data: (network, recsys) -> round -> fraction of posts recommended
rec_frac_runs = defaultdict(list)

for network in networks:
    for recsys in recsyss:
        key = (network, recsys)
        print(f"Processing {network} - {recsys}")

        for run in runs:
            db_path = f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"

            round_frac = defaultdict(float)

            with sqlite3.connect(db_path) as conn:
                cursor = conn.cursor()
                
                # get total posts per round
                cursor.execute("SELECT round, COUNT(*) FROM post GROUP BY round")
                total_posts_per_round = {r: c for r, c in cursor.fetchall()}

                # get number of recommended posts per round (split post_ids string)
                cursor.execute("SELECT round, post_ids FROM recommendations")
                rec_posts_per_round = defaultdict(set)
                for r, post_ids_str in cursor.fetchall():
                    if post_ids_str:
                        rec_posts = {int(pid) for pid in post_ids_str.split("|")}
                        rec_posts_per_round[r].update(rec_posts)

            round_frac = defaultdict(list)  # <-- must be list, not float

            for r in total_posts_per_round.keys():
                frac = len(rec_posts_per_round.get(r, [])) / total_posts_per_round[r]
                round_frac[r].append(frac)  # now this works

            rec_frac_runs[key].append(round_frac)

# --- average and std over runs ---
rows = []
for key, run_rounds in rec_frac_runs.items():
    network, recsys = key

    # collect all rounds
    all_rounds = sorted({r for rr in run_rounds for r in rr.keys()})
    for r in all_rounds:
        vals = [rr.get(r, 0) for rr in run_rounds]
        avg = np.mean(vals)
        std = np.std(vals)
        rows.append({
            "network": network,
            "recsys": recsys,
            "round": r,
            "avg_fraction_recommended": avg,
            "std_fraction_recommended": std
        })

df_rec_fraction = pd.DataFrame(rows)

In [ ]:
# --- Prepare figure: 2 rows x 2 columns ---
fig, axes = plt.subplots(2, figsize=(20, 10), sharex=True)

pbar = tqdm(total=len(networks)*len(recsyss), desc="Processing Experiments", unit="experiment")

for i, network in enumerate(networks):
    ax_frac = axes[i]   # left column for fraction

    for j, recsys in enumerate(recsyss):
        key = (network, recsys)
        
        # --- fraction of posts recommended ---
        to_plot = df_rec_fraction[
            (df_rec_fraction['network'] == network) & 
            (df_rec_fraction['recsys'] == recsys)
        ]
        x = to_plot['round'].values
        y = to_plot['avg_fraction_recommended'].values
        err = to_plot['std_fraction_recommended'].values
        
        line = ax_frac.plot(x + j*0.05, y, 'o-', markersize=2, label=f"{recsys}")[0]
        ax_frac.fill_between(x + j*0.05, y-err, y+err, color=line.get_color(), alpha=0.2)
        
        pbar.update(1)

    # --- labels & titles ---
    ax_frac.set_title(f"Network: {network} - Fraction Recommended")
    ax_frac.set_ylabel("Avg Fraction Recommended")
    ax_frac.set_xlabel("Round")
    ax_frac.legend(loc="upper left")

plt.tight_layout()
plt.show()


# How many reactions does a post receives?

In [ ]:
from tqdm.notebook import tqdm
import sqlite3
from collections import defaultdict
import numpy as np

# --- Online Welford mean/std ---
def update_online(stats, val):
    stats["count"] += 1
    delta = val - stats["mean"]
    stats["mean"] += delta / stats["count"]
    delta2 = val - stats["mean"]
    stats["M2"] += delta * delta2

def finalize_online(stats):
    if stats["count"] < 2:
        return stats["mean"], 0.0
    return stats["mean"], np.sqrt(stats["M2"] / (stats["count"]-1))

# --- Initialize incremental stats ---
reactions_per_post_stats = defaultdict(lambda: {"count":0, "mean":0.0, "M2":0.0})
reactions_before_stats = defaultdict(lambda: {"count":0, "mean":0.0, "M2":0.0})
reactions_after_stats = defaultdict(lambda: {"count":0, "mean":0.0, "M2":0.0})
recommendation_prob_stats = defaultdict(lambda: defaultdict(lambda: {"count":0, "mean":0.0, "M2":0.0}))

pbar = tqdm(total=len(networks)*len(recsyss)*len(runs), desc="Processing Experiments")

# --- Process run by run ---
for network in networks:
    for recsys in recsyss:
        key = (network, recsys)
        for run in runs:
            print(f"Processing {network}-{recsys}-run{run}")
            db_path = f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"
            with sqlite3.connect(db_path) as conn:
                cursor = conn.cursor()

                # --- Fetch all post reactions (streamed one by one to save memory) ---
                cursor.execute("SELECT id, reaction_count FROM post")
                post_reactions_stream = cursor.fetchall()  # list of (id, rc)

                # Update all reactions per post
                for _, rc in post_reactions_stream:
                    update_online(reactions_per_post_stats[key], rc)

                # --- Recommended posts per round ---
                cursor.execute("SELECT round, post_ids FROM recommendations")
                rec_posts_per_round = defaultdict(set)
                for r, post_ids_str in cursor.fetchall():
                    if post_ids_str:
                        rec_posts_per_round[r].update(int(pid) for pid in post_ids_str.split("|"))

                # --- Reactions before/after recommendation ---
                for post_ids in rec_posts_per_round.values():
                    for pid in post_ids:
                        cursor.execute("SELECT reaction_count FROM post WHERE id=?", (pid,))
                        row = cursor.fetchone()
                        if row:
                            rc = row[0]
                            update_online(reactions_before_stats[key], rc)
                            update_online(reactions_after_stats[key], rc)

                # --- Probability to be recommended given prior reactions ---
                for pid, rc in post_reactions_stream:
                    rec_flag = any(pid in post_ids for post_ids in rec_posts_per_round.values())
                    update_online(recommendation_prob_stats[key][rc], rec_flag)

            # free memory
            del post_reactions_stream, rec_posts_per_round

            pbar.update(1)

In [ ]:
# Prepare reactions per post
rows_reactions_per_post = []
rows_before = []
rows_after = []

for key in reactions_per_post_stats:
    network, recsys = key

    # Reactions per post (all)
    mean_all, std_all = finalize_online(reactions_per_post_stats[key])
    rows_reactions_per_post.append({
        "network": network,
        "recsys": recsys,
        "mean": mean_all,
        "std": std_all
    })

    # Reactions before recommendation
    mean_before, std_before = finalize_online(reactions_before_stats[key])
    rows_before.append({
        "network": network,
        "recsys": recsys,
        "mean": mean_before,
        "std": std_before
    })

    # Reactions after recommendation
    mean_after, std_after = finalize_online(reactions_after_stats[key])
    rows_after.append({
        "network": network,
        "recsys": recsys,
        "mean": mean_after,
        "std": std_after
    })

df_reactions_per_post = pd.DataFrame(rows_reactions_per_post)
df_before = pd.DataFrame(rows_before)
df_after = pd.DataFrame(rows_after)


In [ ]:
networks_order = ["BA", "ER"]  # make sure this matches your dataset

fig, axes = plt.subplots(
    nrows=len(networks_order), 
    ncols=3, 
    figsize=(18, 10), 
    sharex=True, 
    sharey=True
)

for i, network in enumerate(networks_order):
    for j, kind in enumerate(["all", "before", "after"]):
        ax = axes[i, j]
        for recsys in recsyss:
            if kind == "all":
                df_plot = df_reactions_per_post[(df_reactions_per_post['network']==network) & 
                                                (df_reactions_per_post['recsys']==recsys)]
            elif kind == "before":
                df_plot = df_before[(df_before['network']==network) & 
                                    (df_before['recsys']==recsys)]
            else:  # after
                df_plot = df_after[(df_after['network']==network) & 
                                   (df_after['recsys']==recsys)]
            
            ax.bar(df_plot['recsys'], df_plot['mean'], yerr=df_plot['std'], 
                   capsize=5, alpha=0.7, label=recsys)
        
        # Titles for first row
        if i == 0:
            if kind == "all":
                ax.set_title("Reactions per post (all posts)")
            elif kind == "before":
                ax.set_title("Reactions before recommendation")
            else:
                ax.set_title("Reactions after recommendation")
        
        # X-axis label only for bottom row
        if i == len(networks_order)-1:
            ax.set_xlabel("Recommendation system")
        # Y-axis label only for first column
        if j == 0:
            ax.set_ylabel(f"Reactions ({network})")
        
        ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
networks_order = ["BA", "ER"]

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(10, 10), sharex=True, sharey=True)

for i, network in enumerate(networks_order):
    ax = axes[i]
    for recsys in recsyss:
        key = (network, recsys)
        rc_vals = sorted(recommendation_prob_stats[key].keys())
        y_avg = []
        y_std = []
        for rc in rc_vals:
            mean, std = finalize_online(recommendation_prob_stats[key][rc])
            y_avg.append(mean)
            y_std.append(std)
        ax.errorbar(rc_vals, y_avg, yerr=y_std, fmt='o-', label=f"{recsys}")

    ax.set_title(f"Recommendation probability vs prior reactions ({network})")
    ax.set_ylabel("P(recommended)")
    ax.set_xticks([i for i in range(30)])
    ax.grid(visible=True, which='major', axis='both')
    ax.legend()

axes[-1].set_xlabel("Reactions in prior rounds")
plt.tight_layout()
plt.show()


In [ ]:
from collections import defaultdict
import sqlite3
import numpy as np

# --- Store reactions per number of recommendations ---
reactions_vs_nrec_stats = defaultdict(lambda: {"count":0, "mean":0.0, "M2":0.0})

pbar = tqdm(total=len(networks)*len(recsyss)*len(runs), desc="Processing Experiments")


for network in networks:
    for recsys in recsyss:
        key = (network, recsys)
        for run in runs:
            db_path = f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"
            with sqlite3.connect(db_path) as conn:
                cursor = conn.cursor()

                # Count number of recommendations per post
                cursor.execute("SELECT post_ids FROM recommendations")
                rec_count = defaultdict(int)
                for (post_ids_str,) in cursor.fetchall():
                    if post_ids_str:
                        for pid in post_ids_str.split("|"):
                            rec_count[int(pid)] += 1

                # Get total reactions per post
                cursor.execute("SELECT id, reaction_count FROM post")
                for pid, rc in cursor.fetchall():
                    nrec = rec_count.get(pid, 0)
                    # Update Welford online stats: reactions per number of recommendations
                    stats = reactions_vs_nrec_stats[(network, recsys, nrec)]
                    stats["count"] += 1
                    delta = rc - stats["mean"]
                    stats["mean"] += delta / stats["count"]
                    delta2 = rc - stats["mean"]
                    stats["M2"] += delta * delta2
            pbar.update(1)

            del rec_count


rows = []
for (network, recsys, nrec), stats in reactions_vs_nrec_stats.items():
    mean, std = finalize_online(stats)
    rows.append({
        "network": network,
        "recsys": recsys,
        "nrec": nrec,
        "mean_reactions": mean,
        "std_reactions": std
    })

df_reactions_vs_nrec = pd.DataFrame(rows)


In [ ]:
group1 = ["F", "FP"]
group2 = ["P", "RC"]

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(14, 10))
networks_order = ["BA", "ER"]

for i, network in enumerate(networks_order):
    for j, recsys_group in enumerate([group1, group2]):
        ax = axes[i, j]  # <- correct indexing for 2x2 array
        for recsys in recsys_group:
            df_plot = df_reactions_vs_nrec[
                (df_reactions_vs_nrec['network']==network) &
                (df_reactions_vs_nrec['recsys']==recsys)
            ].sort_values('nrec')

            ax.plot(df_plot['nrec'], df_plot['mean_reactions'], 'o-', alpha=0.7, label=recsys)
            ax.fill_between(
                df_plot['nrec'],
                df_plot['mean_reactions'] - df_plot['std_reactions'],
                df_plot['mean_reactions'] + df_plot['std_reactions'],
                alpha=0.2
            )

        ax.set_title(f"{network} - {' & '.join(recsys_group)}")
        ax.set_xlabel("Number of recommendations")
        ax.set_ylabel("Average reactions")
        ax.legend(title="Recsys")

plt.tight_layout()
plt.show()


# Degree analysis

In [ ]:
import glob 

output_folder = "processed_recommendations"
os.makedirs(output_folder, exist_ok=True)

avg_rec_per_degree = dict()  # (network, recsys) -> dict: degree -> avg recommendations

pbar = tqdm(total=len(networks)*len(recsyss)*len(runs), desc="Processing Experiments")

for network in networks:
    for recsys in recsyss:
        key = (network, recsys)
        print(f"Processing {network} - {recsys}")
        
        degree_counts_runs = defaultdict(list)
        
        for run in runs:
            folder = f"experiments_recsys/{network}_{recsys}_{run}"
            db_path = f"{folder}/database_server.db"

            # --- find the CSV file in the folder ---
            csv_files = glob.glob(os.path.join(folder, "*.csv"))
            if len(csv_files) != 1:
                raise ValueError(f"Expected exactly 1 CSV in {folder}, found {len(csv_files)}")
            network_csv = csv_files[0]

            
            # --- load agent network ---
            G = nx.read_edgelist(network_csv, delimiter=",")  # assuming CSV: source,target
            degrees = dict(G.degree())  # username -> degree
            
            # --- load post table ---
            with sqlite3.connect(db_path) as conn:
                df_posts = pd.read_sql("SELECT id AS post_id, user_id AS author_id, round, reaction_count FROM Post", conn)
                
                # map author_id to username (assuming user_id in post matches id in user_mgmt)
                df_users = pd.read_sql("SELECT id AS user_id, username FROM user_mgmt", conn)
                df_posts = df_posts.merge(df_users, left_on="author_id", right_on="user_id", how="left")
                
                # map username to degree
                df_posts["author_degree"] = df_posts["username"].map(degrees).fillna(0).astype(int)
                
                # --- load recommendations table ---
                df_recs = pd.read_sql("SELECT user_id as receiver_id, round, post_ids FROM recommendations", conn)
                
                # explode post_ids
                df_recs = df_recs[df_recs["post_ids"].notna()]
                df_recs = df_recs.assign(post_id=df_recs["post_ids"].str.split("|")).explode("post_id")
                df_recs["post_id"] = df_recs["post_id"].astype(int)
                
                # join with posts to get author_degree
                df_recs_full = df_recs.merge(
                    df_posts[["post_id", "author_id", "author_degree", "round"]],
                    left_on=["post_id", "round"],
                    right_on=["post_id", "round"],
                    how="left"
                )
                
                # select needed columns
                df_recs_full = df_recs_full[["author_id", "receiver_id", "post_id", "round", "author_degree"]]
                
                # --- save exploded table ---
                out_csv = os.path.join(output_folder, f"{network}_{recsys}_{run}.csv")
                df_recs_full.to_csv(out_csv, index=False)
                
                # --- compute recommendations per author degree ---
                counts = df_recs_full.groupby("author_degree").size()
                for deg, c in counts.items():
                    degree_counts_runs[deg].append(c)
            
            pbar.update(1)
        
        # --- average over runs ---
        avg_counts = {deg: np.mean(vals) for deg, vals in degree_counts_runs.items()}
        std_counts = {deg: np.std(vals) for deg, vals in degree_counts_runs.items()}
        avg_rec_per_degree[key] = {"avg": avg_counts, "std": std_counts}


In [ ]:
fig, axes = plt.subplots(2, sharey=True, figsize=(12, 6))

for i, network in enumerate(networks):
    ax = axes[i]
    
    for recsys in recsyss:
        key = (network, recsys)
        avg_hist = avg_rec_per_degree[key]["avg"]
        std_hist = avg_rec_per_degree[key]["std"]
        
        degrees_sorted = sorted(avg_hist.keys())
        avg_vals = [avg_hist[d] for d in degrees_sorted]
        std_vals = [std_hist[d] for d in degrees_sorted]
        
        # small horizontal shift to separate overlapping points
        j = recsyss.index(recsys)
        x = np.array(degrees_sorted) + j*0.1
        
        line = ax.plot(x, avg_vals, 'o-', label=recsys)[0]
        ax.fill_between(x, np.array(avg_vals)-np.array(std_vals), np.array(avg_vals)+np.array(std_vals),
                        color=line.get_color(), alpha=0.2)
    
    ax.set_xlabel("Receiver degree in agent network")
    ax.set_ylabel("Average # recommendations")
    ax.set_title(f"Network: {network}")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()


essentially the recommendations received follow the degree distribution

# Debugging code for script

In [39]:
import os
import glob
import sqlite3
import gc
import pickle
import numpy as np
import pandas as pd
from matplotlib import rcParams
import matplotlib.pyplot as plt
from cycler import cycler
from collections import defaultdict, Counter
from tqdm import tqdm

# --- Configuration & Plotting Styles ---
NETWORKS = ['BA', 'ER']
RECSYSS = ['F', 'FP', 'P', 'RC']
RUNS = list(range(10))

# Binning config for Degree plots (aggregates noisy degree data into bins)
DEGREE_BINS = np.linspace(0, 200, 21) # 20 bins from 0 to 200

# Create directories
DIRS = {
    'data': 'processed_data',
    'figs': 'figs'
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

plt.style.use("ggplot")

rcParams.update({

    # ---- FONT ----
    "font.family": "sans-serif",
    "font.style": "normal",

    # ---- FIGURE ----
    "figure.facecolor": "white",
    "figure.dpi": 300,

    "savefig.bbox": "tight",
    "savefig.dpi": 300,
    "savefig.transparent": True,

    # ---- AXES ----
    "axes.facecolor": "white",
    "axes.edgecolor": "black",
    "axes.linewidth": 1,
    "axes.labelsize": 11,
    "axes.labelcolor": "black",

    "axes.spines.right": False,
    "axes.spines.top": False,

    # ---- GRID (delicate dashed grey) ----
    "axes.grid": True,
    "grid.color": "#D3D3D3",
    "grid.linestyle": "--",
    "grid.linewidth": 0.6,
    "grid.alpha": 0.7,

    # ---- TICKS (bigger labels) ----
    "xtick.color": "black",
    "ytick.color": "black",
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.width": 1,
    "ytick.major.width": 1,
    "xtick.major.size": 3,
    "ytick.major.size": 3,

    # ---- LINES ----
    "lines.linewidth": 1.5,
    "lines.markersize": 6,

    # ---- LEGEND ----
    "legend.fontsize": 8,

    # ---- COLOR CYCLE ----
    "axes.prop_cycle": cycler(
        color=["#D81B60", "#1E88E5", "#FFC107", "#004D40"]
    ),
})


# =============================================================================
# PART 1: CORE PROCESSING ENGINE (ETL)
# =============================================================================

def get_db_path(network, recsys, run):
    return f"experiments_recsys/{network}_{recsys}_{run}/database_server.db"

def get_cache_path(network, recsys, run):
    return os.path.join(DIRS['data'], f"{network}_{recsys}_{run}.pkl")

In [59]:
import networkx as nx

def compute_author_degrees_from_csv(csv_path):
    """
    Reads an edge list CSV and computes node degrees using NetworkX.
    Returns a dict: node -> adjusted degree (degree/2)
    """
    try:
        # Read edge list
        df_edges = pd.read_csv(csv_path, header=None, names=['u', 'v'])
        
        # Create undirected graph
        G = nx.from_pandas_edgelist(df_edges, source='u', target='v')
        
        # Compute degrees and divide by 2 as per your note
        author_degrees = {node: deg for node, deg in G.degree()}
        print(author_degrees)
        return author_degrees
    except Exception as e:
        print(f"Error computing degrees from {csv_path}: {e}")
        return {}



In [63]:
from collections import Counter

def plot_degree(author_degrees, network, recsys, run):
    """
    Plots log–log scatter plot of author degrees ('o-') for a given network-recsys-run.
    """
    if not author_degrees:
        print(f"No degree data for {network}-{recsys}-{run}")
        return
    
    degrees = list(author_degrees.values())
    
    # Count frequencies
    counts_dict = Counter(degrees)
    
    # Sort by degree
    degs_sorted = sorted(counts_dict.keys())
    counts_sorted = [counts_dict[d] for d in degs_sorted]
    
    # Plot log–log scatter
    plt.figure(figsize=(6, 4))
    plt.scatter(degs_sorted, counts_sorted, marker='o')
    plt.plot(degs_sorted, counts_sorted, 'o-', alpha=0.7)  # connect points with lines
    
    plt.xscale('log')
    plt.yscale('log')
    plt.xlabel("Author Degree")
    plt.ylabel("Frequency")
    plt.title(f"Degree Distribution: {network}-{recsys}-{run}")
    plt.grid(True, which='both', ls='--', lw=0.5, alpha=0.7)
    
    # Save figure
    filename = f"degree_{network}_{recsys}_{run}.png"
    plt.savefig(os.path.join(DIRS['figs'], filename), dpi=300, facecolor='white')
    plt.close()

In [ ]:
"""
Computes ALL metrics for a single run in one go to minimize I/O.
"""

print(f'Start computing metrics for {network}-{recsys}-{run}')


db_path = get_db_path(network, recsys, run)

if not os.path.exists(db_path):
    print(f"Warning: DB not found at {db_path}")
    #return None 

metrics = {}

# --- 1. Load Network Degree (if available) ---
# Assuming CSV is in the same folder as the DB parent
run_folder = os.path.dirname(db_path)
csv_files = glob.glob(os.path.join(run_folder, "*.csv"))
author_degrees = {}

if csv_files:
    try:
        # Optim: Use pandas to read edge list, it's faster than nx.read_edgelist for large files
        if csv_files:
            author_degrees = compute_author_degrees_from_csv(csv_files[0])
        else:
            author_degrees = {}
    except Exception as e:
        print(f"Error reading network CSV for {network}-{recsys}-{run}: {e}")
        
plot_degree(author_degrees, network, recsys, run)

with sqlite3.connect(db_path) as conn:
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    # --- 2. Post Metadata (ID, Author, Reactions) ---
    # Fetch all posts into a DataFrame
    query_posts = "SELECT id, user_id, reaction_count, round FROM post"
    df_posts = pd.read_sql(query_posts, conn)
    df_posts.set_index('id', inplace=True)
    
    # Map authors to usernames to match with degrees (if needed) or just use user_id
    # Assuming author_degrees keys match 'user_id' or 'username'. 
    # The original script mapped author_id -> username -> degree. 
    # Here we assume we can link user_id to the degree map. 
    # note: If degree map uses usernames, we need the user table.
    
    df_users = pd.read_sql("SELECT id, username FROM user_mgmt", conn)
    user_map = df_users.set_index('id')['username'].to_dict()
    
    df_posts['username'] = df_posts['user_id'].map(user_map)
    df_posts['degree'] = df_posts['username'].map(author_degrees).fillna(0).astype(int)

    # --- 3. Recommendations Analysis ---
    cursor.execute("SELECT round, post_ids, user_id FROM recommendations")
    recs_rows = cursor.fetchall()

df_posts.head()

Start computing metrics for BA-RC-0
{'MichaelChambers': 78, 'MarkLee': 29, 'ToddWells': 23, 'JasonFerrell': 141, 'JoshuaMckinney': 46, 'KennethChaney': 52, 'HeatherWhite': 92, 'StephanieKelly': 74, 'TimothyTaylor': 46, 'DavidFloyd': 54, 'WesleySimmons': 47, 'AaronBrown': 40, 'RobertJennings': 53, 'SamuelKnox': 31, 'ShawnBecker': 50, 'KristenBanks': 27, 'Mrs.TracyGoodwinMD': 25, 'PaulMoore': 13, 'BenjaminHoward': 49, 'PeterDunn': 14, 'JosephMueller': 26, 'LisaRogers': 18, 'ShannonKnight': 14, 'ThomasMcfarland': 22, 'TerryStokes': 24, 'JudithAnderson': 15, 'CrystalMorris': 12, 'SeanCrawford': 15, 'MichaelJones': 14, 'AntonioCarr': 15, 'ChristinaJones': 10, 'VanessaFlores': 10, 'RichardBell': 13, 'GeraldBlanchard': 20, 'HarryNealPhD': 8, 'ErnestHawkins': 8, 'RebeccaBailey': 17, 'EmilyJohnson': 9, 'MarkFreeman': 8, 'RhondaFarmer': 17, 'ValerieLutz': 11, 'JacobBoyer': 7, 'JenniferSmith': 15, 'RobertGutierrez': 12, 'WilliamBaldwin': 8, 'WilliamFields': 9, 'JacobWu': 7, 'BrendaJensen': 8, 'Jo

,user_id,reaction_count,round,username,degree
id,,,,,
1,835,4,1,ShannonHenderson,5
2,835,1,1,ShannonHenderson,5
3,231,2,1,RobertKennedy,10
4,302,1,1,BrianTaylor,10
5,302,2,1,BrianTaylor,10


In [73]:
for row in recs_rows[:5]:
    print(row.keys())

['round', 'post_ids', 'user_id']
['round', 'post_ids', 'user_id']
['round', 'post_ids', 'user_id']
['round', 'post_ids', 'user_id']
['round', 'post_ids', 'user_id']


In [74]:
# Process Recommendations
recs_per_round = defaultdict(int)
post_rec_counts = Counter() # How many times a post was recommended (histogram)
post_rec_rounds = defaultdict(list) # post_id -> [list of rounds]
post_unique_reach = defaultdict(set) # Set of unique users who saw the post

total_recs_count = 0

for row in recs_rows:
    r = row['round']
    viewer_id = row['user_id']
    pids_str = row['post_ids']
    
    if not pids_str:
        recs_per_round[r] += 0
        continue
        
    pids = [int(x) for x in pids_str.split("|")]
    count = len(pids)
    recs_per_round[r] += count
    total_recs_count += count
    
    for pid in pids:
        post_rec_counts[pid] += 1
        post_rec_rounds[pid].append(r)
        post_unique_reach[pid].add(viewer_id)

# Convert sets to counts
post_reach_counts = {pid: len(viewers) for pid, viewers in post_unique_reach.items()}

# --- Metric A: Recommendations per Round (Time Series) ---
metrics['recs_per_round'] = dict(recs_per_round)

# --- Metric B: Recommendation Frequency Histogram (Recs per Post) ---
# We store the Counter of counts: {1 rec: 50 posts, 2 recs: 30 posts...}
metrics['rec_count_hist'] = dict(Counter(post_rec_counts.values()))

# --- Metric C: Consecutive Streaks & Inter-event Gaps ---
streak_counts = Counter()
gap_counts = Counter()
consecutive_pairs = 0
total_pairs = 0

for pid, rounds in post_rec_rounds.items():
    rounds.sort()
    
    # Streaks
    current_streak = 1
    for i in range(1, len(rounds)):
        if rounds[i] == rounds[i-1] + 1:
            current_streak += 1
            consecutive_pairs += 1
        else:
            streak_counts[current_streak] += 1
            current_streak = 1
            
            # Gap
            gap = rounds[i] - rounds[i-1] - 1
            gap_counts[gap] += 1
            
        total_pairs += 1
    streak_counts[current_streak] += 1 # Capture the last streak
    
metrics['streak_hist'] = dict(streak_counts)
metrics['gap_hist'] = dict(gap_counts)
metrics['consec_prob'] = consecutive_pairs / total_pairs if total_pairs > 0 else 0

# --- Metric D: Fraction of Total Posts Recommended ---
# Group posts by round to get total posts per round
total_posts_per_round = df_posts['round'].value_counts().to_dict()

# Unique posts recommended per round
unique_recs_per_round = defaultdict(set)
for row in recs_rows:
    if row['post_ids']:
        pids = [int(x) for x in row['post_ids'].split("|")]
        unique_recs_per_round[row['round']].update(pids)
        
frac_rec_per_round = {}
for r, total_p in total_posts_per_round.items():
    if total_p > 0:
        frac_rec_per_round[r] = len(unique_recs_per_round.get(r, [])) / total_p
    else:
        frac_rec_per_round[r] = 0.0
metrics['frac_rec_per_round'] = frac_rec_per_round

# --- Metric E: Reactions Analysis ---
# 1. Reactions vs # Recommendations
# df_posts already has 'reaction_count'. We map 'rec_count' to it.
df_posts['rec_count'] = df_posts.index.map(post_rec_counts).fillna(0).astype(int)

# Group by number of recommendations -> average reactions
reactions_by_rec_count = df_posts.groupby('rec_count')['reaction_count'].mean().to_dict()
metrics['reactions_by_rec_count'] = reactions_by_rec_count

# 2. Reactions Before vs After (Approximation based on logic)
# The original script fetched reactions individually. 
# Since 'reaction_count' in Post table is likely the *final* count, 
# distinguishing before/after strictly requires a 'Reaction' table with timestamps.
# Assuming the original script logic: it compared "reactions of recommended posts" vs "all posts".
# We will store:
#   - Avg reactions of ALL posts
#   - Avg reactions of RECOMMENDED posts (at least once)
metrics['avg_reactions_all'] = df_posts['reaction_count'].mean()
metrics['avg_reactions_rec'] = df_posts[df_posts['rec_count'] > 0]['reaction_count'].mean()

# --- Metric F: Recommendations vs Author Degree ---
# df_posts has 'degree' and 'rec_count'.
# We want total recommendations received by authors of degree X.
# Note: A post has one author. If post X is recommended 5 times, that author gets 5 rec credits.
rec_by_degree = df_posts.groupby('degree')['rec_count'].sum()
# Normalize by number of authors with that degree? 
# Original script: "Average # recommendations". 
# Usually means: (Total Recs received by posts from authors of degree k) / (Number of authors of degree k)
# OR (Total Recs received by posts from authors of degree k) / (Number of posts from authors of degree k)
# Let's align with typical analysis: Avg recs per post for a given author degree.

avg_rec_per_degree = df_posts.groupby('degree')['rec_count'].mean().to_dict()
metrics['rec_per_degree'] = avg_rec_per_degree

# --- Metric G: Unique Reach vs Author Degree ---
df_posts['unique_reach'] = df_posts.index.map(post_reach_counts).fillna(0).astype(int)

In [75]:
df_posts

,user_id,reaction_count,round,username,degree,rec_count,unique_reach
id,,,,,,,
1,835,4,1,ShannonHenderson,5,8,4
2,835,1,1,ShannonHenderson,5,8,4
3,231,2,1,RobertKennedy,10,10,4
4,302,1,1,BrianTaylor,10,10,4
5,302,2,1,BrianTaylor,10,10,4
...,...,...,...,...,...,...,...
91230,234,1,1440,JacobBoyer,7,4,1
91231,685,0,1440,SheenaPhelps,6,0,0
91232,685,0,1440,SheenaPhelps,6,0,0


In [76]:
# Group by degree and take MEAN unique reach
# We use a binned approach later, but storing raw means here is flexible
reach_by_degree = df_posts.groupby('degree')['unique_reach'].mean().to_dict()
metrics['reach_per_degree'] = reach_by_degree

print(f'End computing metrics for {network}-{recsys}-{run}')

End computing metrics for BA-RC-0


In [78]:
metrics.keys()

dict_keys(['recs_per_round', 'rec_count_hist', 'streak_hist', 'gap_hist', 'consec_prob', 'frac_rec_per_round', 'reactions_by_rec_count', 'avg_reactions_all', 'avg_reactions_rec', 'rec_per_degree', 'reach_per_degree'])

Starting Rich-get-Richer Analysis...


Processing:   1%|▏         | 1/80 [00:10<13:36, 10.33s/it]

KeyboardInterrupt: 

In [2]:
import os
import glob
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm import tqdm

# =============================================================================
# CONFIGURATION
# =============================================================================

BASE_DIR = "experiments_recsys"
NETWORKS = ['BA', 'ER']
RECSYSS = ['F', 'FP', 'P', 'RC']
RUNS = list(range(10))

# Output directory
FIG_DIR = "figs_quality_time"
os.makedirs(FIG_DIR, exist_ok=True)

# Plot Styling
plt.style.use('ggplot')
plt.rcParams.update({
    'font.size': 12,
    'figure.figsize': (10, 6),
    'axes.spines.right': False,
    'axes.spines.top': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'savefig.transparent': False
})

# =============================================================================
# CORE LOGIC
# =============================================================================

def get_db_path(network, recsys, run):
    # Adjust this path pattern if your folder structure is slightly different
    return os.path.join(BASE_DIR, f"{network}_{recsys}_{run}", "database_server.db")

def compute_quality_time_series(network, recsys, run):
    """
    Computes the average reaction count of posts recommended at each round.
    Returns: dict {round: avg_reaction_count}
    """
    db_path = get_db_path(network, recsys, run)
    if not os.path.exists(db_path):
        return None

    with sqlite3.connect(db_path) as conn:
        conn.row_factory = sqlite3.Row
        
        # 1. Load Post Quality (Final Reaction Count)
        # We assume 'reaction_count' in the post table reflects the total/final popularity.
        # This tells us if the system is recommending "good" (popular) content at round T.
        df_posts = pd.read_sql("SELECT id, reaction_count FROM post", conn)
        post_quality = df_posts.set_index('id')['reaction_count'].to_dict()
        
        # 2. Stream Recommendations
        cursor = conn.cursor()
        cursor.execute("SELECT round, post_ids FROM recommendations")
        
        # Storage: round -> list of qualities of recommended posts
        round_qualities = defaultdict(list)
        
        while True:
            rows = cursor.fetchmany(10000)
            if not rows:
                break
            
            for row in rows:
                r = row['round']
                pids_str = row['post_ids']
                
                if not pids_str:
                    continue
                
                try:
                    pids = [int(x) for x in pids_str.split('|')]
                except ValueError:
                    continue
                
                # For every recommended post, get its quality (reaction count)
                for pid in pids:
                    q = post_quality.get(pid, 0)
                    round_qualities[r].append(q)

    # 3. Compute Average per Round
    avg_quality_per_round = {}
    for r, qualities in round_qualities.items():
        if qualities:
            avg_quality_per_round[r] = np.mean(qualities)
        else:
            avg_quality_per_round[r] = 0.0
            
    return avg_quality_per_round

# =============================================================================
# AGGREGATION & PLOTTING
# =============================================================================

def aggregate_time_series(list_of_dicts):
    """
    Aggregates a list of runs (dicts) into Mean and Std arrays for plotting.
    """
    if not list_of_dicts:
        return None, None, None
    
    # Find all rounds present across runs
    all_rounds = set()
    for d in list_of_dicts:
        all_rounds.update(d.keys())
    
    if not all_rounds:
        return None, None, None
        
    sorted_rounds = sorted(list(all_rounds))
    
    means = []
    stds = []
    
    for r in sorted_rounds:
        # Collect values for round r from all runs
        vals = [d[r] for d in list_of_dicts if r in d]
        if vals:
            means.append(np.mean(vals))
            stds.append(np.std(vals))
        else:
            means.append(0)
            stds.append(0)
            
    return np.array(sorted_rounds), np.array(means), np.array(stds)

if __name__ == "__main__":
    print("Starting Analysis: Quality of Recommendations over Time...")
    
    # Data Storage: data[network][recsys] = list of runs
    data = defaultdict(lambda: defaultdict(list))
    
    total_ops = len(NETWORKS) * len(RECSYSS) * len(RUNS)
    pbar = tqdm(total=total_ops, desc="Processing Runs")
    
    # 1. Collect Data
    for network in NETWORKS:
        for recsys in RECSYSS:
            for run in RUNS:
                res = compute_quality_time_series(network, recsys, run)
                if res:
                    data[network][recsys].append(res)
                pbar.update(1)
    pbar.close()
    
    # 2. Plotting
    print("Generating Plot...")
    
    fig, axes = plt.subplots(1, len(NETWORKS), figsize=(14, 6))
    if len(NETWORKS) == 1: axes = [axes]
    
    for i, network in enumerate(NETWORKS):
        ax = axes[i]
        
        for recsys in RECSYSS:
            runs = data[network][recsys]
            x, y, err = aggregate_time_series(runs)
            
            if x is not None:
                ax.plot(x, y, label=recsys, linewidth=2)
                ax.fill_between(x, y - err, y + err, alpha=0.15)
        
        ax.set_title(f"{network} - Quality of Recommendations")
        ax.set_xlabel("Round (Time)")
        ax.set_ylabel("Avg Reactions of Recommended Posts")
        ax.legend(loc='upper left')
        ax.grid(True, linestyle='--', alpha=0.5)
        
    plt.tight_layout()
    output_path = os.path.join(FIG_DIR, "avg_reactions_of_recs_over_time.png")
    plt.show()
    print(f"Saved plot to {output_path}")

Starting Analysis: Quality of Recommendations over Time...


Processing:   1%|▏         | 1/80 [05:26<7:09:26, 326.16s/it]


KeyboardInterrupt: 